In [ ]:
from pathlib import Path
import hashlib
import io
import json
import stat
import subprocess
import sys
import zipfile
from google.colab import files

EXPECTED_ZIP_SHA256 = "4a44144878283f9bef078871dcc472fef35dcf4dd54b29595fa1dfa1bba2f53d"
EXPECTED_FREEZE_MANIFEST_SHA256 = "973123c9c7ef06780a2dfc9f3f77d2441d4ddd2fddc2d535296296aeca32c7f8"
runtime_root = Path("/content/task_A_test_runtime")

def verify_existing_runtime(root):
    from pathlib import PurePosixPath
    manifest_path = root / "bundle_manifest.json"
    expected_manifest_sha = "4e9bbaa0e033a75047ba00c7924e352335ce61627015c15f5274eda2ff38d95c"
    if not manifest_path.is_file() or hashlib.sha256(manifest_path.read_bytes()).hexdigest() != expected_manifest_sha:
        raise RuntimeError("The release list of the current directory does not match. Please keep the directory and the error message, do not overwrite the file or continue the reasoning.")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected_files = set(manifest["sha256"]) | {"bundle_manifest.json"}
    actual_files = set()
    for path in root.rglob("*"):
        if path.is_symlink() or not path.resolve().is_relative_to(root.resolve()):
            raise RuntimeError("The current running directory contains symbolic links or paths that exceed the boundaries.")
        if path.is_file() and "__pycache__" not in path.relative_to(root).parts:
            actual_files.add(path.relative_to(root).as_posix())
    if actual_files != expected_files:
        raise RuntimeError("The files in the current running directory are incomplete or contain additional files. Please keep the directory and the error message, do not overwrite them.")
    for name, wanted in manifest["sha256"].items():
        relative = PurePosixPath(name)
        if relative.is_absolute() or ".." in relative.parts or "\\" in name or ":" in name:
            raise RuntimeError("The package list contains unsafe paths.")
        path = root.joinpath(*relative.parts)
        if path.is_symlink() or not path.resolve().is_relative_to(root.resolve()):
            raise RuntimeError("The file path for running is abnormal.")
        if hashlib.sha256(path.read_bytes()).hexdigest() != wanted:
            raise RuntimeError(f"The running file is different from the distribution version.：{name}。Please keep the table of contents and error messages, and do not overwrite them.")
    frozen_manifest = root / "frozen/freeze_manifest.json"
    if hashlib.sha256(frozen_manifest.read_bytes()).hexdigest() != EXPECTED_FREEZE_MANIFEST_SHA256:
        raise RuntimeError("A The frozen version does not match.")

def upload_runtime():
    print("Please choose CS5494_TaskA_Final_Test_Colab.zip.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Only choose one ZIP.")
    uploaded_name, archive_bytes = next(iter(uploaded.items()))
    if hashlib.sha256(archive_bytes).hexdigest() != EXPECTED_ZIP_SHA256:
        raise ValueError("Please check the selected ZIP。")

    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        members = archive.infolist()
        names = [item.filename for item in members]
        if not members or len(members) > 128 or len(names) != len(set(name.casefold() for name in names)):
            raise ValueError("ZIP file error。")
        if sum(item.file_size for item in members) > 15 * 1024 * 1024:
            raise ValueError("ZIP unzip size error。")
        targets = []
        for item in members:
            name = item.filename
            parts = name.split("/")
            if (not name.startswith("task_A_test_runtime/") or "\\" in name
                    or ":" in name or ".." in parts or name.startswith("/")
                    or item.flag_bits & 1
                    or stat.S_IFMT(item.external_attr >> 16) == stat.S_IFLNK):
                raise ValueError(f"unsafe ZIP ：{name}")
            relative = Path(*parts[1:])
            if relative.name.casefold().startswith("test_answer"):
                raise ValueError("No test answer file.")
            target = runtime_root / relative
            if not target.resolve().is_relative_to(runtime_root.resolve()):
                raise ValueError("ZIP tagert path exceeded.")
            targets.append((item, target))
        if archive.testzip() is not None:
            raise ValueError("ZIP CRC Verification failed.")
        runtime_root.mkdir(parents=True, exist_ok=True)
        for item, target in targets:
            if item.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(item))

if runtime_root.exists() and not runtime_root.is_dir():
    raise RuntimeError("The running path has been occupied by a file. Please keep the file and check the path.")
if runtime_root.is_dir() and any(runtime_root.iterdir()):
    verify_existing_runtime(runtime_root)
    print("The same official test run package has been found and verified. Skipping the duplicate upload:", runtime_root)
else:
    upload_runtime()
    verify_existing_runtime(runtime_root)
    print("The package has been verified and decompressed:", runtime_root)
print("This unit is only for preparing or restoring files; it will not automatically install or generate model responses.")
print("First time setup complete: Proceed with environment installation; Installed and restarted: Jump to 'Restart and continue from here' for environment check.")
print("If all the predictions have been completed previously, please proceed with the integrity check and scoring without running the reasoning again.")


已找到并校验相同的正式测试运行包，跳过重复上传： /content/task_A_test_runtime
本单元只准备或恢复文件；不会自动安装或生成模型回答。
首次准备完成：继续环境安装；已安装并重启：跳到“重启后从这里继续”的环境检查。
如果此前已完成全部预测，请继续完整性检查和评分，不要再次运行推理。


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

runtime_root = Path("/content/task_A_test_runtime")
if not (runtime_root / "frozen/environment_lock.json").is_file():
    raise RuntimeError("请先运行上面的 ZIP 上传和解压单元。")

torch_packages = {
    "torch": "2.11.0+cu130",
    "torchvision": "0.26.0+cu130",
    "torchaudio": "2.11.0+cu130",
}
def installed_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

if any(installed_version(name) != wanted for name, wanted in torch_packages.items()):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        *[f"{name}=={wanted}" for name, wanted in torch_packages.items()],
        "--index-url", "https://download.pytorch.org/whl/cu130",
    ])
requirements = runtime_root / "frozen/project/common/requirements_inference.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "-r", str(requirements)])
marker = {"installer_pid": os.getpid(), "restart_required": True,
          "reason": "Reload torch/CUDA and Hugging Face packages in a fresh Python kernel"}
Path("/content/task_A_environment_install.json").write_text(json.dumps(marker), encoding="utf-8")
print("安装完成。现在手动重启会话；重连后从下方环境检查单元继续，不要重新运行上传或安装单元。")


安装完成。现在手动重启会话；重连后从下方环境检查单元继续，不要重新运行上传或安装单元。


In [ ]:
# 重启后从这里继续；重新建立后续单元需要的变量。
from pathlib import Path
import json
import os
import subprocess
import sys
from google.colab import files

runtime_root = Path("/content/task_A_test_runtime")
marker_path = Path("/content/task_A_environment_install.json")
if not marker_path.is_file():
    raise RuntimeError("No installation records were found. Please complete the ZIP upload and environment installation first.")
marker = json.loads(marker_path.read_text(encoding="utf-8"))
if marker["installer_pid"] == os.getpid():
    raise RuntimeError("After installation, the Python session has not been restarted yet. Please manually restart it and then re-run this check unit.")

sys.path.insert(0, str(runtime_root))
from runner_helpers import verify_bundle, verify_freeze
verify_bundle(runtime_root)
verify_freeze(runtime_root)
subprocess.check_call([sys.executable, str(runtime_root / "frozen/verify_freeze.py"), "--check-runtime"])
import torch
import transformers
import accelerate
import huggingface_hub
import tokenizers
import safetensors

lock = json.loads((runtime_root / "frozen/environment_lock.json").read_text(encoding="utf-8"))
actual = {"torch": str(torch.__version__), "cuda": torch.version.cuda,
          "transformers": transformers.__version__, "accelerate": accelerate.__version__,
          "huggingface-hub": huggingface_hub.__version__, "tokenizers": tokenizers.__version__,
          "safetensors": safetensors.__version__}
expected = {"torch": lock["torch"], "cuda": lock["cuda"], **lock["critical_packages"]}
differences = {key: {"expected": expected[key], "actual": actual[key]}
               for key in expected if actual[key] != expected[key]}
if differences:
    raise RuntimeError(f"Currently, the actual module of the Notebook is still different from the frozen environment:{differences}")
if not torch.cuda.is_available() or torch.cuda.get_device_name(0) != lock["gpu"]:
    raise RuntimeError("Please select an available T4 GPU session.")
probe = torch.ones((2, 2), device="cuda", dtype=torch.float16)
assert (probe @ probe).tolist() == [[2.0, 2.0], [2.0, 2.0]]
torch.cuda.synchronize()
del probe
print(json.dumps({"status": "FROZEN_ENVIRONMENT_READY", **actual,
                  "gpu": torch.cuda.get_device_name(0), "cuda_operation_passed": True}, indent=2))


{
  "status": "FROZEN_ENVIRONMENT_READY",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "transformers": "5.20.2",
  "accelerate": "1.15.0",
  "huggingface-hub": "1.33.0",
  "tokenizers": "0.23.3",
  "safetensors": "0.8.0",
  "gpu": "Tesla T4",
  "cuda_operation_passed": true
}


In [ ]:
%run -i /content/task_A_test_runtime/run_colab_task_A_test.py


GPU: Tesla T4
Loading a fresh pinned FP16 model for Task A; no prior notebook model is reused.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Checking token lengths for all 1007 final-test inputs.
{
  "status": "TASK_A_TEST_LENGTH_PREFLIGHT_PASSED",
  "count": 1007,
  "ready_for_generation": true,
  "over_budget_count": 0,
  "target_truncation": false,
  "reference_answers_read": false,
  "input_budget": 3840,
  "output_budget": 256,
  "min": 59,
  "median": 117,
  "mean": 204.59880834160873,
  "max": 655,
  "per_task_max": {
    "rag": 655,
    "math": 149,
    "qa": 111
  }
}
[1/1007] rag TEST_00001 | 0.519s | answer='Sons for the Return Home'
[2/1007] math TEST_00002 | 1.374s | answer='8026'
[3/1007] rag TEST_00003 | 0.371s | answer='UNANSWERABLE'
[4/1007] qa TEST_00004 | 0.270s | answer='The Revenant'
[5/1007] rag TEST_00005 | 0.390s | answer='Arsenal_F.C.'
[6/1007] qa TEST_00006 | 0.766s | answer='information that can be passed to a requester includes'
[7/1007] math TEST_00007 | 0.297s | answer='35.0'
[8/1007] rag TEST_00008 | 0.372s | answer='UNANSWERABLE'
[9/1007] math TEST_00009 | 0.219s | answer='144'
[10/1007] math

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
run_dir = Path(TASK_A_TEST_RUN_DIR)
stage1_summary = json.loads((run_dir / "runner_summary.json").read_text(encoding="utf-8"))
if (stage1_summary.get("status") != "TASK_A_TEST_INFERENCE_COMPLETE_PENDING_SCORING"
        or stage1_summary.get("completed_count") != 1007
        or not (run_dir / "inference/predictions.csv").is_file()):
    raise RuntimeError("The first stage has not been fully successful yet, so the reference answers cannot be uploaded for formal grading.")
print(json.dumps(stage1_summary, ensure_ascii=False, indent=2))
print("Phase 1 ZIP：", TASK_A_TEST_PREDICTION_ZIP)
print("Please save the downloaded prediction ZIP file; if the browser failed to download it, you can run the following line.")
# files.download(TASK_A_TEST_PREDICTION_ZIP)


{
  "status": "TASK_A_TEST_INFERENCE_COMPLETE_PENDING_SCORING",
  "scope": "All 1007 final-test inputs; A frozen baseline",
  "freeze_id": "TaskA_baseline_fp16_evaluation_v2_20261027_v1",
  "count": 1007,
  "completed_count": 1007,
  "final_test_frozen": true,
  "gpu_execution": "completed",
  "reference_answers_read": false,
  "reference_reads_during_inference": 0,
  "task_counts": {
    "rag": 350,
    "math": 327,
    "qa": 330
  },
  "environment_checks": {
    "status": "passed",
    "fresh_pinned_fp16_model": true,
    "critical_dependencies_match": true,
    "snapshot_matches": true,
    "all_generation_parameters_match": true
  },
  "generation_receipt_sha256": "99448f3b8143002a892cfe48bc71c8e94063fc4519d53274d53f4b53d8398258",
  "blank_raw_answers": 0,
  "empty_extracted_answers": 65,
  "truncated_outputs": 0,
  "max_input_tokens": 655,
  "summed_generation_seconds": 515.3765225099951,
  "generation_wall_seconds": 518.2690932479995,
  "gpu_memory": {
    "peak_allocated_gib": 

In [ ]:
stage1_summary = json.loads((Path(TASK_A_TEST_RUN_DIR) / "runner_summary.json").read_text(encoding="utf-8"))
if (stage1_summary.get("status") != "TASK_A_TEST_INFERENCE_COMPLETE_PENDING_SCORING"
        or stage1_summary.get("completed_count") != 1007):
    raise RuntimeError("If the first stage is not successful, uploading of the reference answers is not permitted.")
sys.path.insert(0, str(runtime_root))
from runner_helpers import check_generation_complete
completion_check = check_generation_complete(runtime_root, Path(TASK_A_TEST_RUN_DIR))
print("The complete prediction and freeze version has been rechecked:", completion_check["status"])
print("Now select dataset/test_answer.csv。")
reference_upload = files.upload()
if len(reference_upload) != 1:
    raise ValueError("Select only one test reference answer CSV。")
reference_name = next(iter(reference_upload))
reference_path = str(Path(reference_name).resolve())
if not Path(reference_path).is_file():
    raise FileNotFoundError(reference_path)
print("The reference answers have been uploaded. Now, let's proceed to run the scoring module.")


完整预测及冻结版本已再次校验： passed
现在请选择 dataset/test_answer.csv。


Saving test_answer.csv to test_answer.csv
参考答案已上传，接下来运行评分单元。


In [ ]:
%run -i /content/task_A_test_runtime/score_task_A_test.py --run-dir "$TASK_A_TEST_RUN_DIR" --references "$reference_path"


{
  "status": "TASK_A_FINAL_TEST_PASSED",
  "freeze_id": "TaskA_baseline_fp16_evaluation_v2_20261027_v1",
  "count": 1007,
  "generation_completion_verified_before_reference_access": true,
  "reference_answers_read": true,
  "reference_sha256": "2b760d80e44ff18d399d14ddf97bedeebadf7221d54c3a89f7cb236838086fda",
  "scope": "All 1007 test examples; frozen Task A baseline",
  "final_test_frozen": true,
  "gpu_execution": "completed",
  "task_counts": {
    "rag": 350,
    "math": 327,
    "qa": 330
  },
  "completed_count": 1007,
  "freeze_manifest_sha256": "973123c9c7ef06780a2dfc9f3f77d2441d4ddd2fddc2d535296296aeca32c7f8",
  "scoring_directory": "scoring_20261027_095659_433343_UTC",
  "blank_raw_answers": 0,
  "empty_extracted_answers": 65,
  "truncated_outputs": 0,
  "max_input_tokens": 655,
  "generation_wall_seconds": 518.2690932479995,
  "gpu_memory": {
    "peak_allocated_gib": 3.37813663482666,
    "peak_reserved_gib": 4.173828125,
    "ending_allocated_gib": 3.2127695083618164
  }

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>